# Lightweight HBCC — PlantVillage Training Pipeline

Trains all models on the PlantVillage plant disease dataset (38 classes, ~87 K images).
Images are resized to 32×32 to match the HBCC CIFAR configuration exactly.

**Required Kaggle dataset:** `vipoooool/new-plant-diseases-dataset`  
Add it to this notebook before running. On Kaggle's filesystem it mounts at:
```
/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/
  New Plant Diseases Dataset(Augmented)/
    New Plant Diseases Dataset(Augmented)/   ← nested folder
      train/
      valid/
```

**Three sessions required** (PlantVillage is ~1.6× more batches/epoch than CIFAR-10):

| Session | What runs | Est. time |
|---------|-----------|----------|
| **A** | ResNet-18 (teacher) + MobileNetV2 + ShuffleNetV2 + CoC | ~7.5 h |
| **B** | HBCC-Small CE + HBCC-Medium CE | ~7.5 h |
| **C** | ResNet-18 (KD teacher) + HBCC-Small KD + HBCC-Medium KD + benchmarks | ~10.5 h |

Enable **only** the phases for the current session before committing.

In [ ]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Repo root not found: {ROOT}"

PYTHON = Path(sys.executable)
print("Repo:", ROOT)
print("Python:", PYTHON)

## Phase Switches

**Session A:** set `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.  
**Session B:** set `RUN_STUDENT_CE` = True; all others False.  
**Session C:** set `RUN_TEACHER_KD`, `RUN_KD`, `RUN_BENCHMARKS` = True; all others False.

> **Cross-session accuracy:** each session saves a partial metrics CSV to `results/`.  
> In Session C, fill in `PREV_SESSION_A_METRICS` and `PREV_SESSION_B_METRICS` with the  
> paths to those CSVs (downloaded or added as a Kaggle dataset input) so the final  
> publication table includes all 8 models.

In [ ]:
# ── Session A ──────────────────────────────────────────────────────────────
RUN_ENV_CHECKS            = True   # recommended in Session A only
RUN_TEACHER               = True   # ResNet-18 reference baseline
RUN_LIGHTWEIGHT_BASELINES = True   # MobileNetV2, ShuffleNetV2
RUN_COC_BASELINE          = True   # CoC pure-cluster baseline

# ── Session B ──────────────────────────────────────────────────────────────
RUN_STUDENT_CE            = False  # HBCC-Small and HBCC-Medium CE-only

# ── Session C ──────────────────────────────────────────────────────────────
RUN_TEACHER_KD            = False  # Re-train ResNet-18 teacher for KD
RUN_KD                    = False  # HBCC KD variants
RUN_BENCHMARKS            = False  # Latency / params / MACs for all models

# ── Epoch budgets ──────────────────────────────────────────────────────────
FULL_EPOCHS     = 100   # ResNet-18, MobileNetV2, ShuffleNetV2, CoC
ACCURACY_EPOCHS = 150   # HBCC CE and KD variants

# ── PlantVillage dataset paths ─────────────────────────────────────────────
# Kaggle filesystem path (includes username prefix for this dataset).
_PV_ROOT    = "/kaggle/input/datasets/vipoooool/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
PLANT_TRAIN = f"{_PV_ROOT}/train"
PLANT_VALID = f"{_PV_ROOT}/valid"

PLANTVILLAGE_BASE = [
    "data.name=plantvillage",
    f"data.train_dir={PLANT_TRAIN}",
    f"data.val_dir={PLANT_VALID}",
    "model.num_classes=38",
    "data.workers=4",
]

# ── Benchmark settings ─────────────────────────────────────────────────────
BENCHMARK_OUTPUT  = "results/benchmark_plantvillage"
BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP  = 30
BENCHMARK_RUNS    = 100

DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP  = 2
DEBUG_BENCHMARK_RUNS    = 3

# ── Cross-session accuracy (Session C only) ────────────────────────────────
# After Session A/B: commit and download results/plantvillage_session_a.csv etc.,
# or add the committed output as a Kaggle dataset input, then fill in the path below.
PREV_SESSION_A_METRICS = ""  # e.g. "/kaggle/input/hbcc-pv-s-a/results/plantvillage_session_a.csv"
PREV_SESSION_B_METRICS = ""  # e.g. "/kaggle/input/hbcc-pv-s-b/results/plantvillage_session_b.csv"

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def pv_overrides(experiment_name: str | None = None, overrides: list[str] | None = None) -> list[str]:
    items = [*PLANTVILLAGE_BASE]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(pv_overrides(experiment_name, overrides))
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup  = DEBUG_BENCHMARK_WARMUP  if debug else BENCHMARK_WARMUP
    runs    = DEBUG_BENCHMARK_RUNS    if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py", "--config", config,
        "--output", BENCHMARK_OUTPUT,
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    args += override_args(pv_overrides(experiment_name))
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"[warn] Checkpoint not found — benchmarking architecture only: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def save_session_metrics(session_label: str, run_dirs: list[str]) -> Path:
    """Collect training metrics from the given run dirs and save as results/<label>.csv."""
    rows = []
    for run_root in run_dirs:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })
    out_dir = ROOT / "results"
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / f"{session_label}.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"Saved {len(rows)} rows → {out_path}")
    return out_path

## 0. Environment Checks

Run only in **Session A** (`RUN_ENV_CHECKS = True`).  
Verifies dataset paths and class count, runs the unit-test suite, and traces HBCC-Med shapes.

In [ ]:
if RUN_ENV_CHECKS:
    # Verify dataset paths before committing hours to training
    for label, path in [("train", PLANT_TRAIN), ("valid", PLANT_VALID)]:
        if not Path(path).is_dir():
            raise FileNotFoundError(
                f"PlantVillage {label} dir not found: {path}\n"
                "Ensure 'vipoooool/new-plant-diseases-dataset' is added as a dataset input."
            )
    classes = sorted(d for d in os.listdir(PLANT_TRAIN) if Path(f"{PLANT_TRAIN}/{d}").is_dir())
    n_train = sum(len(os.listdir(f"{PLANT_TRAIN}/{c}")) for c in classes)
    n_valid = sum(
        len(os.listdir(f"{PLANT_VALID}/{c}"))
        for c in sorted(d for d in os.listdir(PLANT_VALID) if Path(f"{PLANT_VALID}/{d}").is_dir())
    )
    print(f"Classes : {len(classes)}")
    print(f"Train   : {n_train} images")
    print(f"Valid   : {n_valid} images")
    assert len(classes) == 38, f"Expected 38 classes, found {len(classes)}: {classes}"
    print("OK: 38 classes confirmed")

    run_py(["-m", "pytest"])

    run_py([
        "tools/shape_trace.py", "--config", "configs/hbcc_accuracy_medium.yaml",
        "--batch-size", "1",
        *override_args(["model.num_classes=38"]),
    ])

## Session A — Teacher and Lightweight Baselines

All models trained for `FULL_EPOCHS` (100) with full augmentation.

Estimated time on Kaggle T4: **~7.5 h** (well within the 12 h limit).

At the end of this session `results/plantvillage_session_a.csv` is saved automatically.
Download it (or add the committed Session A output as a Kaggle dataset input to Session C)
so the final publication table includes baseline accuracies.

In [ ]:
EXP_RESNET  = "resnet18_pv"
DIR_TEACHER = "runs_pv_teacher"
TEACHER_CHECKPOINT = f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"

if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER,
        EXP_RESNET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_MOBILENET  = "mobilenet_v2_pv"
EXP_SHUFFLENET = "shufflenet_v2_x1_0_pv"
DIR_BASELINES  = "runs_pv_baselines"

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        DIR_BASELINES,
        EXP_MOBILENET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        DIR_BASELINES,
        EXP_SHUFFLENET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_COC = "coc_baseline_pv"
DIR_COC = "runs_pv_coc"
COC_CHECKPOINT = f"{DIR_COC}/{EXP_COC}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        DIR_COC,
        EXP_COC,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
# Save Session A metrics immediately after training completes.
# Download this file, then pass its path as PREV_SESSION_A_METRICS in Session C.
if any([RUN_TEACHER, RUN_LIGHTWEIGHT_BASELINES, RUN_COC_BASELINE]):
    save_session_metrics("plantvillage_session_a", [DIR_TEACHER, DIR_BASELINES, DIR_COC])

## Session B — HBCC CE-only Training

Both HBCC variants trained with cross-entropy only (no KD).

Estimated time on Kaggle T4: **~7.5 h** (well within the 12 h limit).

At the end of this session `results/plantvillage_session_b.csv` is saved automatically.

In [ ]:
EXP_HBCC_SML = "hbcc_accuracy_small_pv"
EXP_HBCC_MED = "hbcc_accuracy_medium_pv"
DIR_ACCURACY = "runs_pv_accuracy"

if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_ACCURACY,
        EXP_HBCC_SML,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_ACCURACY,
        EXP_HBCC_MED,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )

In [ ]:
if RUN_STUDENT_CE:
    save_session_metrics("plantvillage_session_b", [DIR_ACCURACY])

## Session C — KD Training, Benchmarks, and Publication Table

Re-trains ResNet-18 as the KD teacher (in its own output directory so Session A’s
checkpoint is not overwritten), then distils into both HBCC variants, runs all
benchmarks, and builds the final table.

Estimated time on Kaggle T4: **~10.5 h** (safe within 12 h).

> **Tip:** To skip the ~2 h teacher re-train, set `RUN_TEACHER_KD = False` and point
> `TEACHER_KD_CHECKPOINT` to the Session A checkpoint you added as a dataset input:
> ```python
> TEACHER_KD_CHECKPOINT = "/kaggle/input/<your-dataset>/runs_pv_teacher/resnet18_pv/best.pth"
> ```
> This reduces Session C to **~8.5 h**.

In [ ]:
EXP_RESNET_KD         = "resnet18_pv_kd"
DIR_TEACHER_KD        = "runs_pv_teacher_kd"
TEACHER_KD_CHECKPOINT = f"{DIR_TEACHER_KD}/{EXP_RESNET_KD}/best.pth"

# Override if Session A output is available as a dataset input:
# TEACHER_KD_CHECKPOINT = "/kaggle/input/<your-dataset>/runs_pv_teacher/resnet18_pv/best.pth"

if RUN_TEACHER_KD:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER_KD,
        EXP_RESNET_KD,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_HBCC_SML_KD = "hbcc_accuracy_small_kd_pv"
EXP_HBCC_MED_KD = "hbcc_accuracy_medium_kd_pv"
DIR_KD          = "runs_pv_kd"

KD_OVERRIDES = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

if RUN_KD:
    teacher_path = ROOT / TEACHER_KD_CHECKPOINT
    if not teacher_path.exists():
        raise FileNotFoundError(
            f"Teacher checkpoint not found: {teacher_path}\n"
            "Set RUN_TEACHER_KD=True, or point TEACHER_KD_CHECKPOINT to an existing checkpoint."
        )
    print(f"Teacher checkpoint: {teacher_path}")

    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_KD,
        EXP_HBCC_SML_KD,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_KD,
        EXP_HBCC_MED_KD,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )

In [ ]:
if any([RUN_TEACHER_KD, RUN_KD]):
    save_session_metrics("plantvillage_session_c", [DIR_TEACHER_KD, DIR_KD])

## Benchmark Matrix

Measures latency, params, and MACs for all models. Checkpoints from previous sessions
are used if present; if missing (from an earlier Kaggle session), the architecture
metrics (params, MACs, latency) are still measured correctly using random weights.

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",            EXP_RESNET,       f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",        EXP_MOBILENET,    f"{DIR_BASELINES}/{EXP_MOBILENET}/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", EXP_SHUFFLENET,   f"{DIR_BASELINES}/{EXP_SHUFFLENET}/best.pth"),
    ("configs/coc_cifar_baseline.yaml",                  EXP_COC,          COC_CHECKPOINT),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML,     f"{DIR_ACCURACY}/{EXP_HBCC_SML}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED,     f"{DIR_ACCURACY}/{EXP_HBCC_MED}/best.pth"),
    ("configs/hbcc_accuracy_small.yaml",                 EXP_HBCC_SML_KD,  f"{DIR_KD}/{EXP_HBCC_SML_KD}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                EXP_HBCC_MED_KD,  f"{DIR_KD}/{EXP_HBCC_MED_KD}/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print(f"Skip missing config: {cfg}")
            continue
        benchmark(cfg, exp_name, checkpoint=ckpt)

## Results Tables

These cells run in every session and produce whatever data is currently available.

For the **complete publication table** you need all three sessions’ outputs.
In Session C, fill in `PREV_SESSION_A_METRICS` and `PREV_SESSION_B_METRICS` at the top.

In [ ]:
PV_RUN_DIRS = [
    DIR_TEACHER, DIR_BASELINES, DIR_COC, DIR_ACCURACY,
    DIR_TEACHER_KD, DIR_KD,
]


def collect_train_metrics() -> pd.DataFrame:
    """Collect from current-session run dirs, then augment with previous-session CSVs."""
    rows = []
    for run_root in PV_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record  = test_records[-1] if test_records else {}
            rows.append({
                "experiment":    metrics_path.parent.name,
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "test_acc1":     test_record.get("test_acc1"),
                "test_acc5":     test_record.get("test_acc5"),
            })

    _cols = ["experiment", "best_val_acc1", "best_val_acc5", "test_acc1", "test_acc5"]
    df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)

    for prev_path in [PREV_SESSION_A_METRICS, PREV_SESSION_B_METRICS]:
        if not prev_path:
            continue
        p = Path(prev_path)
        if p.exists():
            prev_df = pd.read_csv(p)
            existing = set(df["experiment"])
            new_rows = prev_df[~prev_df["experiment"].isin(existing)]
            df = pd.concat([df, new_rows], ignore_index=True)
            print(f"Merged {len(new_rows)} rows from {p.name}")
        else:
            print(f"[warn] Previous-session CSV not found (skipped): {p}")

    if df.empty:
        return df
    return df.sort_values("best_val_acc1", ascending=False).reset_index(drop=True)


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        try:
            rows.append(json.loads(path.read_text(encoding="utf-8")))
        except json.JSONDecodeError:
            print(f"[warn] Could not parse benchmark JSON: {path.name}")
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
arch_cols = ["config_id", "model_name", "params_total", "macs", "latency_ms_b1", "peak_memory_mb"]
bench_df[[c for c in arch_cols if c in bench_df.columns]] if not bench_df.empty else bench_df

In [ ]:
def build_publication_table(bench: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        print("No benchmark data yet — run Session C with RUN_BENCHMARKS=True.")
        return bench
    out = bench.copy()
    out["acc1"] = None
    out["acc5"] = None
    if not train.empty:
        acc1_key = "test_acc1" if "test_acc1" in train.columns else "best_val_acc1"
        acc5_key = "test_acc5" if "test_acc5" in train.columns else "best_val_acc5"
        acc1_map = dict(zip(train["experiment"], train[acc1_key].fillna(train["best_val_acc1"])))
        acc5_map = dict(zip(train["experiment"], train[acc5_key])) if acc5_key in train.columns else {}
        for idx, row in out.iterrows():
            name = row.get("config_id")
            if name in acc1_map:
                out.at[idx, "acc1"] = acc1_map[name]
            if name in acc5_map:
                out.at[idx, "acc5"] = acc5_map[name]
    rename = {
        "config_id":      "Model",
        "acc1":           "Top-1 Acc (%)",
        "acc5":           "Top-5 Acc (%)",
        "params_total":   "Params",
        "macs":           "MACs",
        "latency_ms_b1":  "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    table = out[[c for c in rename if c in out.columns]].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


pub_df = build_publication_table(bench_df, train_df)
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
pub_df.to_csv(out_dir / "plantvillage_published_results.csv", index=False)
pub_df